In [4]:
import os
import sys
from dataclasses import dataclass

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.get_objective import get_objective

In [6]:
import json
n_trial = "44"
study_name = "xgb-023"
params_id = f"trl{n_trial}"
params_path = f"../../artifacts/optuna/{study_name}/{params_id}.json"
with open(params_path, "r") as f:
    manifest = json.load(f)

params = manifest
print(params)

{'learning_rate': 0.02, 'max_depth': 17, 'min_child_weight': 54.39645499969017, 'colsample_bytree': 0.3335644011630281, 'subsample': 0.8736838639084301, 'reg_alpha': 0.2366545277095805, 'reg_lambda': 9.922363245478374, 'nthread': 1}


In [7]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "xgb"
    data_id: str = "044"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"


cfg = Config()

opts = {
    "earlys_stopping_rounds": 5,
    "max_epochs": 20,
    "min_epochs": 4,
}

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc


True

In [12]:
# === Build & Run (frozen) ===
# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


create_objective = get_objective(cfg.model_name)
objective = create_objective(
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    n_jobs=1,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params={"x": params}
)

[I 2025-10-02 08:43:11,848] Using an existing study with name 'xgb-044' instead of creating a new one.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 10.81 GB
Free GPU Mem: 6.8 GB
[0]	train-auc:0.96218	valid-auc:0.96186
[100]	train-auc:0.97324	valid-auc:0.97237
[200]	train-auc:0.97496	valid-auc:0.97373
[300]	train-auc:0.97617	valid-auc:0.97458
[400]	train-auc:0.97710	valid-auc:0.97513
[500]	train-auc:0.97785	valid-auc:0.97546
[600]	train-auc:0.97846	valid-auc:0.97569
[700]	train-auc:0.97898	valid-auc:0.97585
[800]	train-auc:0.97946	valid-auc:0.97595
[900]	train-auc:0.97987	valid-auc:0.97604
[1000]	train-auc:0.98033	valid-auc:0.97613
[1100]	train-auc:0.98070	valid-auc:0.97618
[1200]	train-auc:0.98108	valid-auc:0.97623
[1300]	train-auc:0.98143	valid-auc:0.97627
[1400]	train-auc:0.98178	valid-auc:0.97631
[1500]	train-auc:0.98211	valid-auc:0.97635
[1600]	train-auc:0.98243	valid-auc:0.97637
[1700]	train-auc:0.98277	valid-auc:0.97642
[1800]	train-auc:0.98306	valid-auc:0.97644
[1900]	train-auc:0.98335	valid-auc:0.97645
[2000]	train-auc:0.98365	valid-auc:0.97648
[2100]	train-auc:0.98394	valid-auc:0.97650
[2

iter_f1,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
train/f1/auc,▁▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███
valid/f1/auc,▁▄▄▅▆▇▇▇▇▇▇▇▇▇▇▇████████████████████████
auc_f1,0.97662
best_iter_f1,3920
iter_f1,4420
runtime_f1,2.4052
train/f1/auc,0.98941
valid/f1/auc,0.9766


[I 2025-10-02 08:45:51,766] Trial 4 finished with value: 0.9766214189311406 and parameters: {'learning_rate': 0.02, 'max_depth': 8, 'min_child_weight': 95.07143064099162, 'colsample_bytree': 0.592797576724562, 'subsample': 0.7394633936788146, 'reg_alpha': 0.0007482139197236472, 'reg_lambda': 0.000602521573620386}. Best is trial 4 with value: 0.9766214189311406.
✅ Message sent.
